# Sequential Chain

串联式调用语言模型（将一个调用的输出作为另一个调用的输入）。

顺序链（Sequential Chain ）允许用户连接多个链并将它们组合成执行特定场景的流水线（Pipeline）。有两种类型的顺序链：

把多个 LLM 处理步骤串成一条流水线：
上一步的输出 → 自动变成下一步的输入
一步步自动执行，不用人工干预。

它解决的问题：
一次提问要经过多轮模型处理，比如：先总结文本再翻译成英文再提炼关键词
这种多步骤流程，就必须用 Sequential Chain。

新版优先用管道语法，旧 LLMChain/SequentialChain 都已弃用。


## 旧版 SimpleSequentialChain && SequentialChain
- SimpleSequentialChain：
- 输入 → 步骤1输出 → 步骤2输入 → 步骤2输出 → 步骤3输入...
- from langchain.chains import SimpleSequentialChain
    最简单形式的顺序链，每个步骤都具有单一输入/输出，并且一个步骤的输出是下一个步骤的输入。
    每个步骤只有 1 个输入、1 个输出
    输出直接丢给下一步，不能自定义变量名
    只能线性无脑传递：A → B → C
    适合：简单流水线、单输入单输出。
  
- SequentialChain：更通用形式的顺序链，允许多个输入/输出。
- from langchain.chains import SequentialChain
    支持多个输入、多个输出
    可以给每个输入 / 输出自定义变量名
    可以选择哪些输出传给下一步
    企业真实场景 90% 用这个
    适合：复杂业务、多参数流转、多步骤处理。

    输入：{article}
    步骤1输出：{summary}
    步骤2输入：{summary}
    步骤2输出：{english_summary}
    步骤3输入：{english_summary}
    步骤3输出：{keywords}
    最终输出：{summary}, {english_summary}, {keywords}

## 新版已经没有这两个类了，不靠类名区分，靠「写法形式」区分，完全对应旧版两种能力。
- 新版 直管道连写 p1|llm|p2|llm
    ➜ 等价旧版 SimpleSequentialChain

- 新版 RunnablePassthrough + .assign 存中间变量
    ➜ 等价旧版 SequentialChain（完整版、多输入多输出）




In [1]:
# ======================  导入依赖 ======================
from langchain_core.prompts import PromptTemplate  # 提示词模板
from langchain_community.chat_models import ChatZhipuAI        # 智谱模型
from langchain_core.output_parsers import StrOutputParser  # 输出解析器
from langchain.chains import LLMChain,SequentialChain         
import os
from dotenv import load_dotenv
load_dotenv()


# ======================  初始化 智谱模型 ======================
llm = ChatZhipuAI(model="glm-4.5-air", api_key=os.getenv("ZHIPUAI_API_KEY"), temperature=0.3)

# ====================== 步骤1：总结文章 ======================
prompt1 = PromptTemplate(
    template="请总结下面内容：\n{article}",
    input_variables=["article"]
)
chain1 = LLMChain(llm=llm, prompt=prompt1, output_key="summary")
chain1 = LLMChain(llm=llm, prompt=prompt1, output_key="summary")

# ====================== 步骤2：翻译成英文 ======================
prompt2 = PromptTemplate(
    template="把这段总结翻译成英文：\n{summary}",
    input_variables=["summary"]
)
chain2 = LLMChain(llm=llm, prompt=prompt2, output_key="english_summary")

# ====================== 步骤3：提取关键词 ======================
prompt3 = PromptTemplate(
    template="从英文总结里提取3个关键词：\n{english_summary}",
    input_variables=["english_summary"]
)
chain3 = LLMChain(llm=llm, prompt=prompt3, output_key="keywords")

# ====================== 组合成顺序链 ======================
final_chain = SequentialChain(
    chains=[chain1, chain2, chain3],     # 三步按顺序执行
    input_variables=["article"],         # 最初输入
    output_variables=[                   
        "summary", "english_summary", "keywords"  # 最终返回所有结果
    ],
    verbose=True
)

# 执行
result = final_chain.invoke({
    "article": "人工智能正在改变各行各业，提高效率，降低成本。"
})

# 输出所有结果
print("\n===== 总结 =====")
print(result["summary"])

print("\n===== 英文总结 =====")
print(result["english_summary"])

print("\n===== 关键词 =====")
print(result["keywords"])


/var/folders/5y/kn_pcqq52hz14rd2rchcpkq80000gn/T/ipykernel_28716/262807534.py:19: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain1 = LLMChain(llm=llm, prompt=prompt1, output_key="summary")




> Entering new SequentialChain chain...

> Finished chain.

===== 总结 =====
人工智能正通过改变各行各业，实现效率提升和成本降低。

===== 英文总结 =====
Artificial Intelligence is transforming various industries, leading to enhanced efficiency and reduced costs.

===== 关键词 =====
Artificial Intelligence, Industries, Efficiency


In [2]:
# ======================  导入依赖 ======================
from langchain_core.prompts import PromptTemplate  # 提示词模板
from langchain_community.chat_models import ChatZhipuAI        # 智谱模型
from langchain_core.runnables import RunnablePassthrough      
import os
from dotenv import load_dotenv
load_dotenv()


# ======================  初始化 智谱模型 ======================
llm = ChatZhipuAI(model="glm-4.5-air", api_key=os.getenv("ZHIPUAI_API_KEY"), temperature=0.3)


# 保留中间变量，用 .assign() 把每一步结果存到字典里，全程保留
# 不保留中间变量：只关心最终结果，中间步骤用完就扔，简洁省内存。
# 保留中间变量：需要查看、使用每一步的输出，业务复杂才用。


# ===================== 步骤1：总结文章 =====================
prompt1 = PromptTemplate.from_template("请总结下面内容：\n{article}")
step1 = prompt1 | llm

# ===================== 步骤2：翻译成英文 =====================
prompt2 = PromptTemplate.from_template("把这段总结翻译成英文：\n{summary}")
step2 = prompt2 | llm

# ===================== 步骤3：提取关键词 =====================
prompt3 = PromptTemplate.from_template("从英文总结里提取3个关键词：\n{english_summary}")
step3 = prompt3 | llm

# ===================== 新版顺序链（官方推荐！） =====================
# 新版用 RunnablePassthrough + assign 实现多变量传递，这是官方替代 SequentialChain 的方案！
sequential_chain = (
    # 第一步：把文章传进去，生成总结 summary
    RunnablePassthrough.assign(summary=lambda x: step1.invoke({"article": x["article"]}))
    
    # 第二步：用 summary 生成英文总结
    .assign(english_summary=lambda x: step2.invoke({"summary": x["summary"]}))
    
    # 第三步：用英文总结生成关键词
    .assign(keywords=lambda x: step3.invoke({"english_summary": x["english_summary"]}))
)


# ===================== 执行 =====================
result = sequential_chain.invoke({
    "article": "人工智能正在改变各行各业，提高效率，降低成本。"
})


# 输出所有结果
print("\n===== 总结 =====")
print(result["summary"])

print("\n===== 英文总结 =====")
print(result["english_summary"])

print("\n===== 关键词 =====")
print(result["keywords"])



===== 总结 =====
content='人工智能正通过改变各行各业，实现效率提升与成本降低。' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 229, 'prompt_tokens': 21, 'prompt_tokens_details': {'cached_tokens': 4}, 'total_tokens': 250}, 'model_name': 'glm-4.5-air', 'finish_reason': 'stop'} id='run-886de508-0325-4c6c-95d5-461db9b37989-0'

===== 英文总结 =====
content='Artificial intelligence is improving efficiency and reducing costs by transforming various industries.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 74, 'prompt_tokens': 124, 'prompt_tokens_details': {'cached_tokens': 4}, 'total_tokens': 198}, 'model_name': 'glm-4.5-air', 'finish_reason': 'stop'} id='run-a641717f-738c-40e8-a6de-2c4ed0207ae8-0'

===== 关键词 =====
content='从提供的英文总结内容 "Artificial intelligence is improving efficiency and reducing costs by transforming various industries." 中，提取的3个关键词如下：\n\n1. **Artificial intelligence**  \n   （核心主题，代表整个句子的主语和焦点。）\n\n2. **efficiency**  \n   （关键作用之一，描述AI如何提升效率。）\n\n